<a href="https://colab.research.google.com/github/Kenny625819/Applied-Data-Science/blob/main/IEEE_code5_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Complete Unified Reproducible Analysis Script
---------------------------------------------

This script integrates the contents of code5, code6, and code7 into a
single Colab-ready script for the 177-case submission-ready analysis.

Included:
1. OOF LightGBM survival prediction (3M / 6M / 12M)
2. ROC + calibration figures
3. SHAP summary plots
4. SHAP mean-absolute heatmap
5. Main performance summary
6. Ablation analysis
7. Temporal validation
8. Spearman correlation
9. ESCC PDP
10. Table3 (ablation) export
11. Table4 (clinical vs expert vs CNN ESCC) export

Input:
    MRI_ALLdata_OOF177.xlsx

Expected columns:
    - ESCC   : CNN-derived ESCC
    - Expert : expert-derived / consensus ESCC

Outputs:
    ./RESULTS_FIGURES_177/
    ./RESULTS_TABLES_177/
"""

# =============================================================================
# Install packages if needed (Colab-safe)
# =============================================================================
import sys
import subprocess

def ensure_package(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])

ensure_package("lightgbm")
ensure_package("shap")
ensure_package("openpyxl")

# =============================================================================
# Imports
# =============================================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import norm, spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# Global settings
# =============================================================================
INPUT_XLSX = "MRI_ALLdata_OOF177.xlsx"

OUT_DIR = Path("RESULTS_FIGURES_177")
OUT_DIR.mkdir(exist_ok=True)

TABLE_OUT_DIR = Path("RESULTS_TABLES_177")
TABLE_OUT_DIR.mkdir(exist_ok=True)

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 20,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 15,
    "figure.titlesize": 24,
    "axes.linewidth": 1.2,
    "lines.linewidth": 2.2,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "savefig.dpi": 600,
    "axes.unicode_minus": False,
})

RANDOM_STATE = 42
N_SPLITS = 5
N_BOOT = 2000

# =============================================================================
# Utility mappings
# =============================================================================
def map_sex(x):
    s = str(x).strip().lower()
    if s in ["1", "m", "male", "男", "男性"]:
        return 1
    if s in ["0", "f", "female", "女", "女性"]:
        return 0
    return np.nan


def map_escc(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        x = str(int(x)) if float(x).is_integer() else str(x)
    s = str(x).strip().lower().replace(" ", "")
    return {"1a": 1, "1b": 2, "1c": 3, "2": 4, "3": 5}.get(s, np.nan)


def frankel_bin(x):
    s = str(x).strip().upper()
    if s in ["A", "B", "C"]:
        return 0
    if s in ["D", "E"]:
        return 1
    return np.nan


def map_yesno(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        if x in [1, 2]:
            return 1
        if x == 0:
            return 0
    s = str(x).strip().lower()
    if s in ["yes", "y", "true", "1", "2", "あり", "有"]:
        return 1
    if s in ["no", "n", "false", "0", "なし", "無"]:
        return 0
    return np.nan


# =============================================================================
# Statistics
# =============================================================================
def auc_ci_bootstrap(y, p, n_boot=N_BOOT, seed=RANDOM_STATE):
    y = np.asarray(y)
    p = np.asarray(p)

    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    aucs = []

    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        try:
            aucs.append(roc_auc_score(y[b], p[b]))
        except ValueError:
            continue

    auc = roc_auc_score(y, p)
    lo, hi = np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)
    return auc, lo, hi


def delong_type_test(y, p1, p2):
    y = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)

    m = int(np.sum(y))
    n = len(y) - m
    if m == 0 or n == 0:
        return np.nan, np.nan

    a1 = roc_auc_score(y, p1)
    a2 = roc_auc_score(y, p2)

    var = (a1 * (1 - a1) / max(n, 1)) + (a2 * (1 - a2) / max(n, 1))
    z = (a1 - a2) / np.sqrt(var + 1e-12)
    p = 2 * (1 - norm.cdf(abs(z)))
    return a1 - a2, p


def get_binary_metrics(y, p, thr=0.5):
    y = np.asarray(y)
    p = np.asarray(p)
    yhat = (p >= thr).astype(int)

    cm = confusion_matrix(y, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    _, _, f1, _ = precision_recall_fscore_support(
        y, yhat, average="binary", zero_division=0
    )
    return sens, spec, f1


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr - fpr)]


def calibration_slope_intercept(y_true, p_pred):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred)

    eps = 1e-6
    p_pred = np.clip(p_pred, eps, 1 - eps)
    logit_p = np.log(p_pred / (1 - p_pred)).reshape(-1, 1)

    lr = LogisticRegression(fit_intercept=True, solver="lbfgs")
    lr.fit(logit_p, y_true)

    slope = float(lr.coef_[0][0])
    intercept = float(lr.intercept_[0])
    return slope, intercept


# =============================================================================
# Model training
# =============================================================================
def run_lgb_oof_calibrated(X, y, n_splits=N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.05,
        num_leaves=31,
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    oof = np.zeros(len(y), dtype=float)

    for tr, te in skf.split(X, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y[tr])
        oof[te] = model.predict_proba(X.iloc[te])[:, 1]

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof, y)
    calibrated = iso.transform(oof)

    final_model = lgb.LGBMClassifier(**params)
    final_model.fit(X, y)

    return calibrated, final_model


def run_lgb_temporal_train_test_calibrated(X_train, y_train, X_test):
    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.05,
        num_leaves=31,
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    p_train = model.predict_proba(X_train)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_train, y_train)

    p_test_raw = model.predict_proba(X_test)[:, 1]
    p_test_cal = iso.transform(p_test_raw)

    return p_test_cal, model, iso


# =============================================================================
# Plotting
# =============================================================================
def plot_roc(ax, y, p_lgb, p_tok, p_kat, title):
    fpr_lgb, tpr_lgb, _ = roc_curve(y, p_lgb)
    fpr_tok, tpr_tok, _ = roc_curve(y, p_tok)
    fpr_kat, tpr_kat, _ = roc_curve(y, p_kat)

    ax.plot(
        fpr_lgb, tpr_lgb,
        color="black", lw=2.5,
        label=f"LightGBM model\n(AUC={roc_auc_score(y, p_lgb):.3f})"
    )
    ax.plot(
        fpr_tok, tpr_tok,
        color="black", ls="--", lw=2.0,
        label=f"Revised Tokuhashi\n(AUC={roc_auc_score(y, p_tok):.3f})"
    )
    ax.plot(
        fpr_kat, tpr_kat,
        color="black", ls=":", lw=2.0,
        label=f"New Katagiri\n(AUC={roc_auc_score(y, p_kat):.3f})"
    )

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.set_title(title, fontsize=24)
    ax.set_xlabel("1 - Specificity", fontsize=22)
    ax.set_ylabel("Sensitivity", fontsize=22)
    ax.tick_params(labelsize=20)
    ax.legend(loc="lower right", frameon=True, fontsize=15)


def plot_calibration(ax, y, p):
    d = pd.DataFrame({"y": y, "p": p})
    d["bin"] = pd.qcut(d["p"], q=10, duplicates="drop")
    g = d.groupby("bin", observed=False).agg(
        obs=("y", "mean"),
        pred=("p", "mean")
    ).reset_index()

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.plot(g["pred"], g["obs"], "o-", color="black", lw=2, markersize=5)
    ax.set_title("Calibration plot", fontsize=24)
    ax.set_xlabel("Predicted probability", fontsize=22)
    ax.set_ylabel("Observed frequency", fontsize=22)
    ax.tick_params(labelsize=20)


def plot_roc_temporal(ax, y, p, title):
    fpr, tpr, _ = roc_curve(y, p)
    auc_val = roc_auc_score(y, p)

    ax.plot(
        fpr, tpr,
        color="black", lw=2.5,
        label=f"LightGBM model\n(AUC={auc_val:.3f})"
    )
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.set_title(title, fontsize=24)
    ax.set_xlabel("1 - Specificity", fontsize=22)
    ax.set_ylabel("Sensitivity", fontsize=22)
    ax.tick_params(labelsize=20)
    ax.legend(loc="lower right", frameon=True, fontsize=15)


def plot_calibration_temporal(ax, y, p):
    d = pd.DataFrame({"y": y, "p": p})
    d["bin"] = pd.qcut(d["p"], q=10, duplicates="drop")
    g = d.groupby("bin", observed=False).agg(
        obs=("y", "mean"),
        pred=("p", "mean")
    ).reset_index()

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.plot(g["pred"], g["obs"], "o-", color="black", lw=2, markersize=5)
    ax.set_title("Calibration plot", fontsize=24)
    ax.set_xlabel("Predicted probability", fontsize=22)
    ax.set_ylabel("Observed frequency", fontsize=22)
    ax.tick_params(labelsize=20)


def shap_summary_plot(model, X, title, filename, rename_map=None):
    X_disp = X.copy()
    if rename_map is not None:
        X_disp = X_disp.rename(columns=rename_map)

    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        sv = sv[1]

    shap.summary_plot(sv, X_disp, show=False)

    fig = plt.gcf()
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=14)
    for t in ax.get_yticklabels():
        t.set_fontsize(14)

    try:
        cbar = fig.axes[-1]
        cbar.tick_params(labelsize=14)
        cbar.set_ylabel(cbar.get_ylabel(), fontsize=14)
    except Exception:
        pass

    plt.title(title, fontsize=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close()


def shap_mean_heatmap(models, X_dict, features, filename, rename_map=None):
    display_features = [rename_map.get(f, f) if rename_map else f for f in features]
    table = pd.DataFrame(index=display_features, columns=models.keys())

    for tag, model in models.items():
        X = X_dict[tag]
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)
        if isinstance(sv, list):
            sv = sv[1]

        abs_mean = np.abs(sv).mean(axis=0)
        for f_orig, f_disp, val in zip(features, display_features, abs_mean):
            table.loc[f_disp, tag] = val

    if "3M" in models:
        table = table.sort_values(by="3M", ascending=False)

    table_num = table.astype(float)

    vmin_fixed = 0.0
    vmax_fixed = 2.5

    # Figure size is kept the same
    fig = plt.figure(figsize=(6.8, 9.5))

    # Heatmap body: widened to about 1.5x, title removed
    ax = fig.add_axes([0.50, 0.06, 0.24, 0.90])

    im = ax.imshow(
        table_num.values,
        cmap="Greys",
        aspect="auto",
        vmin=vmin_fixed,
        vmax=vmax_fixed
    )

    ax.set_xticks(np.arange(len(table.columns)))
    ax.set_xticklabels(table.columns, fontsize=16)
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=16)
    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", labelsize=16)

    threshold = (vmax_fixed - vmin_fixed) * 0.5 + vmin_fixed

    for i in range(table_num.shape[0]):
        for j in range(table_num.shape[1]):
            val = float(table_num.iloc[i, j])
            color = "white" if val > threshold else "black"
            ax.text(
                j, i, f"{val:.3f}",
                ha="center", va="center",
                color=color, fontsize=10
            )

    # Colorbar: nearly same height as heatmap body
    cax = fig.add_axes([0.75, 0.06, 0.022, 0.90])
    cbar = fig.colorbar(
        im,
        cax=cax,
        ticks=np.arange(0.0, 2.5 + 0.001, 0.5)
    )
    cbar.set_label("Mean(|SHAP value|)", fontsize=16)
    cbar.ax.tick_params(labelsize=16)

    # No title
    fig.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close(fig)


def compute_and_plot_pdp(models_by_horizon, X_dict, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    required_tags = ["3M", "6M", "12M"]
    for tag in required_tags:
        if tag not in models_by_horizon:
            raise KeyError(f"Missing model for horizon: {tag}")
        if tag not in X_dict:
            raise KeyError(f"Missing predictor matrix for horizon: {tag}")
        if "ESCC" not in X_dict[tag].columns:
            raise KeyError(f"'ESCC' column not found in X_dict['{tag}']")

    escc_levels = sorted(X_dict["3M"]["ESCC"].dropna().unique())
    if len(escc_levels) == 0:
        raise ValueError("No valid ESCC levels were found in X_dict['3M'].")

    pdp = {}
    for tag in required_tags:
        model = models_by_horizon[tag]
        X_base = X_dict[tag].copy()

        vals = []
        for g in escc_levels:
            X_temp = X_base.copy()
            X_temp["ESCC"] = g
            prob_alive = model.predict_proba(X_temp)[:, 1]
            prob_death = 1.0 - prob_alive
            vals.append(np.mean(prob_death))

        pdp[tag] = np.array(vals)

    label_map = {1: "1a", 2: "1b", 3: "1c", 4: "2", 5: "3"}
    tick_labels = [label_map.get(int(x), str(x)) for x in escc_levels]

    plt.figure(figsize=(8, 6))
    plt.plot(escc_levels, pdp["3M"], color="black", lw=2.8, linestyle="-", marker="o", label="3M mortality")
    plt.plot(escc_levels, pdp["6M"], color="black", lw=2.4, linestyle="--", marker="o", label="6M mortality")
    plt.plot(escc_levels, pdp["12M"], color="black", lw=2.4, linestyle=":", marker="o", label="12M mortality")

    plt.title("Partial dependence of ESCC on predicted mortality risk")
    plt.xlabel("ESCC grade")
    plt.ylabel("Mean predicted mortality risk")
    plt.xticks(escc_levels, tick_labels)

    plt.legend(
        frameon=True,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderpad=0.8
    )

    plt.tight_layout()
    out_path = out_dir / "Supplementary_ESCC_PDP_all_bw.png"
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

    print(f"\nIntegrated PDP figure saved -> {out_path}")


# =============================================================================
# Load data
# =============================================================================
if not Path(INPUT_XLSX).exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_XLSX}")

df_raw = pd.read_excel(INPUT_XLSX)
df_raw = df_raw.loc[:, ~df_raw.columns.str.contains("^Unnamed")].copy()
df_raw.columns = df_raw.columns.str.strip()

required_cols = [
    "Age",
    "Sex",
    "Number of Spinal Metastases",
    "Serum Albumin",
    "CRP",
    "ESCC",
    "Expert",
    "Performance Status (ECOG)",
    "Frankel Grade",
    "Barthel Index (ADL)",
    "Malignancy (Katagiri Score)",
    "Visceral Metastasis (Yes=1/No=0)",
    "Body Mass Index (BMI)",
    "Revised Tokuhashi score",
    "New Katagiri score",
    "3-Month Survival (0=Death, 1=Alive)",
    "6-Month Survival (0=Death, 1=Alive)",
    "12-Month Survival (0=Death, 1=Alive)",
]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns in Excel: {missing}")

df = pd.DataFrame({
    "Age": pd.to_numeric(df_raw["Age"], errors="coerce"),
    "Sex": df_raw["Sex"].apply(map_sex),
    "Number of spinal metastases": pd.to_numeric(df_raw["Number of Spinal Metastases"], errors="coerce"),
    "Albumin": pd.to_numeric(df_raw["Serum Albumin"], errors="coerce"),
    "CRP": pd.to_numeric(df_raw["CRP"], errors="coerce"),
    "ESCC": df_raw["ESCC"].apply(map_escc),
    "ESCC_expert": df_raw["Expert"].apply(map_escc),
    "ECOG": pd.to_numeric(df_raw["Performance Status (ECOG)"], errors="coerce"),
    "Frankel grade": df_raw["Frankel Grade"].apply(frankel_bin),
    "Barthel Index": pd.to_numeric(df_raw["Barthel Index (ADL)"], errors="coerce"),
    "Malignancy (New Katagiri score)": pd.to_numeric(df_raw["Malignancy (Katagiri Score)"], errors="coerce"),
    "Visceral metastasis": df_raw["Visceral Metastasis (Yes=1/No=0)"].apply(map_yesno),
    "BMI": pd.to_numeric(df_raw["Body Mass Index (BMI)"], errors="coerce"),

    "Tokuhashi_binary": (pd.to_numeric(df_raw["Revised Tokuhashi score"], errors="coerce") >= 9).astype(float),
    "Katagiri_binary": (pd.to_numeric(df_raw["New Katagiri score"], errors="coerce") < 7).astype(float),

    "Y_3M": pd.to_numeric(df_raw["3-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_6M": pd.to_numeric(df_raw["6-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_12M": pd.to_numeric(df_raw["12-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
})

print("\n=== Quality check: missing values by analysis column ===")
print(df.isna().sum())

# =============================================================================
# Feature definitions
# =============================================================================
FEATURES = [
    "CRP",
    "Malignancy (New Katagiri score)",
    "Age",
    "Barthel Index",
    "Albumin",
    "BMI",
    "ESCC",
    "ECOG",
    "Number of spinal metastases",
    "Sex",
    "Frankel grade",
    "Visceral metastasis",
]

FEATURE_RENAME = {
    "CRP": "CRP",
    "Malignancy (New Katagiri score)": "Malignancy (New Katagiri score)",
    "Age": "Age",
    "Barthel Index": "Barthel Index",
    "Albumin": "Albumin",
    "BMI": "BMI",
    "ESCC": "ESCC grade",
    "ECOG": "ECOG performance status",
    "Number of spinal metastases": "Number of spinal metastases",
    "Sex": "Sex",
    "Frankel grade": "Frankel grade",
    "Visceral metastasis": "Visceral metastasis",
}

FEATURES_NO_ESCC = [f for f in FEATURES if f != "ESCC"]

BASE_FEATURES = [
    "CRP",
    "Malignancy (New Katagiri score)",
    "Age",
    "Barthel Index",
    "Albumin",
    "BMI",
    "ECOG",
    "Number of spinal metastases",
    "Sex",
    "Frankel grade",
    "Visceral metastasis",
]
FEATURES_EXPERT = BASE_FEATURES + ["ESCC_expert"]
FEATURES_CNN = BASE_FEATURES + ["ESCC"]
FEATURES_CLINICAL = BASE_FEATURES

# =============================================================================
# 1. Spearman correlation
# =============================================================================
corr_sub = df.dropna(subset=["ESCC", "ECOG", "Barthel Index"]).copy()

rho_ecog, p_ecog = spearmanr(corr_sub["ESCC"], corr_sub["ECOG"])
rho_barthel, p_barthel = spearmanr(corr_sub["ESCC"], corr_sub["Barthel Index"])

corr_df_raw = pd.DataFrame([
    {"Variable_1": "ESCC", "Variable_2": "ECOG", "Spearman_rho": rho_ecog, "p_value": p_ecog, "n": len(corr_sub)},
    {"Variable_1": "ESCC", "Variable_2": "Barthel Index", "Spearman_rho": rho_barthel, "p_value": p_barthel, "n": len(corr_sub)},
])
corr_df_raw.to_excel(OUT_DIR / "Spearman_Correlation_177.xlsx", index=False)

corr_df_supp = pd.DataFrame([
    {"Horizon": "3M", "ESCC vs ECOG (ρ)": rho_ecog, "p-value": p_ecog, "ESCC vs Barthel (ρ)": rho_barthel, "p-value (Barthel)": p_barthel, "n": len(corr_sub)},
    {"Horizon": "6M", "ESCC vs ECOG (ρ)": rho_ecog, "p-value": p_ecog, "ESCC vs Barthel (ρ)": rho_barthel, "p-value (Barthel)": p_barthel, "n": len(corr_sub)},
    {"Horizon": "12M", "ESCC vs ECOG (ρ)": rho_ecog, "p-value": p_ecog, "ESCC vs Barthel (ρ)": rho_barthel, "p-value (Barthel)": p_barthel, "n": len(corr_sub)},
])
corr_df_supp.to_excel(OUT_DIR / "Table_S3_Spearman.xlsx", index=False)

print("\n=== Spearman correlation ===")
print(corr_df_raw)

# =============================================================================
# 2. Main survival analysis
# =============================================================================
perf_records = []
models_by_horizon = {}
X_dict = {}

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    sub = df.dropna(subset=FEATURES + [ycol]).copy()
    X = sub[FEATURES]
    y = sub[ycol].astype(int).values

    print(f"\n=== Main analysis: {tag} ===")
    print(f"n = {len(sub)}")

    X_dict[tag] = X

    p_lgb, model_lgb = run_lgb_oof_calibrated(X, y)
    models_by_horizon[tag] = model_lgb

    p_tok = sub["Tokuhashi_binary"].astype(float).values
    p_kat = sub["Katagiri_binary"].astype(float).values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    plot_roc(axes[0], y, p_lgb, p_tok, p_kat, f"{tag} survival")
    plot_calibration(axes[1], y, p_lgb)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"Figure_{tag}_ROC_Calibration_bw.png", dpi=600, bbox_inches="tight")
    plt.close()

    shap_summary_plot(
        model_lgb,
        X,
        f"SHAP summary ({tag})",
        f"SHAP_{tag}_177.png",
        rename_map=FEATURE_RENAME
    )

    lgb_auc, lgb_lo, lgb_hi = auc_ci_bootstrap(y, p_lgb)
    tok_auc, tok_lo, tok_hi = auc_ci_bootstrap(y, p_tok)
    kat_auc, kat_lo, kat_hi = auc_ci_bootstrap(y, p_kat)

    delta_tok, p_lgb_vs_tok = delong_type_test(y, p_lgb, p_tok)
    delta_kat, p_lgb_vs_kat = delong_type_test(y, p_lgb, p_kat)

    thr_lgb = youden_threshold(y, p_lgb)
    sens_lgb, spec_lgb, f1_lgb = get_binary_metrics(y, p_lgb, thr_lgb)
    sens_tok, spec_tok, f1_tok = get_binary_metrics(y, p_tok, 0.5)
    sens_kat, spec_kat, f1_kat = get_binary_metrics(y, p_kat, 0.5)

    brier_lgb = brier_score_loss(y, p_lgb)

    perf_records.append({
        "Horizon": tag,
        "n": len(sub),

        "LightGBM AUC": lgb_auc,
        "LightGBM 95% CI low": lgb_lo,
        "LightGBM 95% CI high": lgb_hi,
        "LightGBM Sensitivity": sens_lgb,
        "LightGBM Specificity": spec_lgb,
        "LightGBM F1": f1_lgb,
        "LightGBM Brier": brier_lgb,

        "Tokuhashi AUC": tok_auc,
        "Tokuhashi 95% CI low": tok_lo,
        "Tokuhashi 95% CI high": tok_hi,
        "Tokuhashi Sensitivity": sens_tok,
        "Tokuhashi Specificity": spec_tok,
        "Tokuhashi F1": f1_tok,

        "Katagiri AUC": kat_auc,
        "Katagiri 95% CI low": kat_lo,
        "Katagiri 95% CI high": kat_hi,
        "Katagiri Sensitivity": sens_kat,
        "Katagiri Specificity": spec_kat,
        "Katagiri F1": f1_kat,

        "Delta AUC (LightGBM - Tokuhashi)": delta_tok,
        "p (LightGBM vs Tokuhashi)": p_lgb_vs_tok,
        "Delta AUC (LightGBM - Katagiri)": delta_kat,
        "p (LightGBM vs Katagiri)": p_lgb_vs_kat,
    })

perf_df = pd.DataFrame(perf_records)
perf_df.to_excel(OUT_DIR / "Performance_Summary_OOF_177.xlsx", index=False)

print("\n=== Performance summary ===")
print(perf_df)

# =============================================================================
# 3. SHAP mean heatmap
# =============================================================================
shap_mean_heatmap(
    models_by_horizon,
    X_dict,
    FEATURES,
    "SHAP_heatmap_3M_6M_12M_177_bw.png",
    rename_map=FEATURE_RENAME,
)

# =============================================================================
# 4. Ablation study
# =============================================================================
ablation_records = []

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    sub = df.dropna(subset=FEATURES + [ycol]).copy()

    X_full = sub[FEATURES]
    X_no = sub[FEATURES_NO_ESCC]
    y = sub[ycol].astype(int).values

    print(f"\n=== Ablation analysis: {tag} ===")
    print(f"n = {len(sub)}")

    p_full, _ = run_lgb_oof_calibrated(X_full, y)
    p_no, _ = run_lgb_oof_calibrated(X_no, y)

    auc_full = roc_auc_score(y, p_full)
    auc_no = roc_auc_score(y, p_no)

    brier_full = brier_score_loss(y, p_full)
    brier_no = brier_score_loss(y, p_no)

    sens_full, spec_full, f1_full = get_binary_metrics(y, p_full, 0.5)
    sens_no, spec_no, f1_no = get_binary_metrics(y, p_no, 0.5)

    delta_auc, p_delong = delong_type_test(y, p_full, p_no)

    ablation_records.append({
        "Horizon": tag,
        "n": len(sub),
        "AUC_Full (clinical + ESCC)": auc_full,
        "AUC_No_ESCC (clinical only)": auc_no,
        "Delta AUC (Full - No_ESCC)": delta_auc,
        "p (DeLong-type)": p_delong,
        "Brier_Full": brier_full,
        "Brier_No_ESCC": brier_no,
        "Sensitivity_Full": sens_full,
        "Specificity_Full": spec_full,
        "F1_Full": f1_full,
        "Sensitivity_No_ESCC": sens_no,
        "Specificity_No_ESCC": spec_no,
        "F1_No_ESCC": f1_no,
    })

ablation_df = pd.DataFrame(ablation_records)
ablation_df.to_excel(OUT_DIR / "Ablation_Study_Output_177.xlsx", index=False)

print("\n=== Ablation study ===")
print(ablation_df)

# =============================================================================
# 5. Temporal validation
# =============================================================================
date_candidates = ["ope date", "ope_date", "Operation Date", "operation date"]
date_col = None
for c in date_candidates:
    if c in df_raw.columns:
        date_col = c
        break

if date_col is None:
    raise ValueError(f"No operation-date column found. Checked: {date_candidates}")

df["ope_date"] = pd.to_datetime(df_raw[date_col], errors="coerce")

print("\n=== Temporal validation setup ===")
print(f"Using date column: {date_col}")
print("Date range:", df["ope_date"].min(), "to", df["ope_date"].max())

cutoff_date = pd.to_datetime("2018-12-31")
temporal_records = []

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    sub = df.dropna(subset=FEATURES + [ycol, "ope_date"]).copy()

    train = sub[sub["ope_date"] <= cutoff_date].copy()
    test = sub[sub["ope_date"] > cutoff_date].copy()

    print(f"\n=== Temporal validation: {tag} ===")
    print(f"Train n = {len(train)}")
    print(f"Test  n = {len(test)}")

    if len(train) == 0 or len(test) == 0:
        print("[SKIP] Empty train or test cohort.")
        continue

    y_train = train[ycol].astype(int).values
    y_test = test[ycol].astype(int).values

    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        print("[SKIP] Need both outcome classes in train and test.")
        continue

    X_train = train[FEATURES]
    X_test = test[FEATURES]

    p_test, model_temp, iso_temp = run_lgb_temporal_train_test_calibrated(
        X_train, y_train, X_test
    )

    auc_temp = roc_auc_score(y_test, p_test)
    brier_temp = brier_score_loss(y_test, p_test)
    slope_temp, intercept_temp = calibration_slope_intercept(y_test, p_test)

    temporal_records.append({
        "Horizon": tag,
        "cutoff_date": str(cutoff_date.date()),
        "n_train": len(train),
        "n_test": len(test),
        "test_events": int(np.sum(y_test)),
        "AUC_temporal": auc_temp,
        "Brier_temporal": brier_temp,
        "Calibration_slope": slope_temp,
        "Calibration_intercept": intercept_temp,
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    plot_roc_temporal(axes[0], y_test, p_test, f"{tag} temporal validation")
    plot_calibration_temporal(axes[1], y_test, p_test)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"Figure_{tag}_Temporal_Validation_bw.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close()

    print(f"Saved: {OUT_DIR / f'Figure_{tag}_Temporal_Validation_bw.png'}")

temporal_df = pd.DataFrame(temporal_records)
temporal_df.to_excel(OUT_DIR / "Temporal_Validation_Summary_177.xlsx", index=False)

print("\n=== Temporal validation summary ===")
print(temporal_df)

# =============================================================================
# 6. PDP
# =============================================================================
compute_and_plot_pdp(models_by_horizon, X_dict, OUT_DIR)

# =============================================================================
# 7. Table 3 / Table 4
# =============================================================================
print("\n=== Generating Table 3 / Table 4 ===")

table3_records = []
table4_records = []

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    # Table 3: ablation
    sub3 = df.dropna(subset=FEATURES + [ycol]).copy()
    y3 = sub3[ycol].astype(int).values

    p_full = run_lgb_oof_calibrated(sub3[FEATURES], y3)[0]
    p_no = run_lgb_oof_calibrated(sub3[FEATURES_NO_ESCC], y3)[0]

    auc_full, auc_full_lo, auc_full_hi = auc_ci_bootstrap(y3, p_full)
    auc_no, auc_no_lo, auc_no_hi = auc_ci_bootstrap(y3, p_no)

    brier_full = brier_score_loss(y3, p_full)
    brier_no = brier_score_loss(y3, p_no)

    delta_auc, p_delong = delong_type_test(y3, p_full, p_no)

    table3_records.append({
        "Horizon": tag,
        "n": len(sub3),
        "AUC_Full": auc_full,
        "AUC_Full_95CI_low": auc_full_lo,
        "AUC_Full_95CI_high": auc_full_hi,
        "AUC_No_ESCC": auc_no,
        "AUC_No_ESCC_95CI_low": auc_no_lo,
        "AUC_No_ESCC_95CI_high": auc_no_hi,
        "Delta_AUC": auc_full - auc_no,
        "p_DeLong_type": p_delong,
        "Brier_Full": brier_full,
        "Brier_No_ESCC": brier_no,
        "Delta_Brier": brier_full - brier_no,
    })

    # Table 4: clinical vs expert vs cnn
    sub4 = df.dropna(subset=FEATURES_CLINICAL + ["ESCC_expert", "ESCC", ycol]).copy()
    y4 = sub4[ycol].astype(int).values

    p_clin = run_lgb_oof_calibrated(sub4[FEATURES_CLINICAL], y4)[0]
    p_exp = run_lgb_oof_calibrated(sub4[FEATURES_EXPERT], y4)[0]
    p_cnn = run_lgb_oof_calibrated(sub4[FEATURES_CNN], y4)[0]

    auc_clin, auc_clin_lo, auc_clin_hi = auc_ci_bootstrap(y4, p_clin)
    auc_exp, auc_exp_lo, auc_exp_hi = auc_ci_bootstrap(y4, p_exp)
    auc_cnn, auc_cnn_lo, auc_cnn_hi = auc_ci_bootstrap(y4, p_cnn)

    brier_clin = brier_score_loss(y4, p_clin)
    brier_exp = brier_score_loss(y4, p_exp)
    brier_cnn = brier_score_loss(y4, p_cnn)

    delta_auc_cnn_exp, p_cnn_vs_exp = delong_type_test(y4, p_cnn, p_exp)

    table4_records.append({
        "Horizon": tag,
        "n": len(sub4),
        "AUC_Clinical": auc_clin,
        "AUC_Clinical_95CI_low": auc_clin_lo,
        "AUC_Clinical_95CI_high": auc_clin_hi,
        "AUC_Expert": auc_exp,
        "AUC_Expert_95CI_low": auc_exp_lo,
        "AUC_Expert_95CI_high": auc_exp_hi,
        "AUC_CNN": auc_cnn,
        "AUC_CNN_95CI_low": auc_cnn_lo,
        "AUC_CNN_95CI_high": auc_cnn_hi,
        "Brier_Clinical": brier_clin,
        "Brier_Expert": brier_exp,
        "Brier_CNN": brier_cnn,
        "Delta_AUC_CNN_minus_Expert": auc_cnn - auc_exp,
        "Delta_Brier_CNN_minus_Expert": brier_cnn - brier_exp,
        "p_DeLong_type_CNN_vs_Expert": p_cnn_vs_exp,
    })

table3_df = pd.DataFrame(table3_records)
table4_df = pd.DataFrame(table4_records)

table3_path = TABLE_OUT_DIR / "Table3_ablation_177_final.xlsx"
table4_path = TABLE_OUT_DIR / "Table4_expert_vs_cnn_177_final.xlsx"

table3_df.to_excel(table3_path, index=False)
table4_df.to_excel(table4_path, index=False)

print(f"Saved: {table3_path}")
print(f"Saved: {table4_path}")

print("\nAll analyses completed successfully.")
print("Figure outputs:", OUT_DIR.resolve())
print("Table outputs :", TABLE_OUT_DIR.resolve())


=== Quality check: missing values by analysis column ===
Age                                0
Sex                                0
Number of spinal metastases        0
Albumin                            0
CRP                                0
ESCC                               0
ESCC_expert                        0
ECOG                               0
Frankel grade                      0
Barthel Index                      0
Malignancy (New Katagiri score)    0
Visceral metastasis                0
BMI                                0
Tokuhashi_binary                   0
Katagiri_binary                    0
Y_3M                               0
Y_6M                               0
Y_12M                              0
dtype: int64

=== Spearman correlation ===
  Variable_1     Variable_2  Spearman_rho   p_value    n
0       ESCC           ECOG      0.072791  0.335631  177
1       ESCC  Barthel Index     -0.095055  0.208206  177

=== Main analysis: 3M ===
n = 177
[LightGBM] [Warning] Found 


=== Main analysis: 6M ===
n = 177
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 83, number of negative: 58
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 183
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f


=== Main analysis: 12M ===
n = 177
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 61, number of negative: 80
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000031 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 184
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f


=== Performance summary ===
  Horizon    n  LightGBM AUC  LightGBM 95% CI low  LightGBM 95% CI high  \
0      3M  177      0.740693             0.663738              0.818036   
1      6M  177      0.814146             0.750127              0.871889   
2     12M  177      0.846558             0.788227              0.899090   

   LightGBM Sensitivity  LightGBM Specificity  LightGBM F1  LightGBM Brier  \
0              0.598540              0.825000     0.725664        0.149418   
1              0.778846              0.712329     0.786408        0.168078   
2              0.753247              0.810000     0.753247        0.152140   

   Tokuhashi AUC  ...  Katagiri AUC  Katagiri 95% CI low  \
0       0.608485  ...      0.665511             0.580959   
1       0.660103  ...      0.692966             0.625593   
2       0.698766  ...      0.702532             0.644662   

   Katagiri 95% CI high  Katagiri Sensitivity  Katagiri Specificity  \
0              0.750824              0.781022

...


=== Ablation study ===
  Horizon    n  AUC_Full (clinical + ESCC)  AUC_No_ESCC (clinical only)  \
0      3M  177                    0.740693                     0.765511   
1      6M  177                    0.814146                     0.824618   
2     12M  177                    0.846558                     0.848896   

   Delta AUC (Full - No_ESCC)  p (DeLong-type)  Brier_Full  Brier_No_ESCC  \
0                   -0.024818         0.796797    0.149418       0.146615   
1                   -0.010472         0.869365    0.168078       0.165710   
2                   -0.002338         0.963304    0.152140       0.154140   

   Sensitivity_Full  Specificity_Full   F1_Full  Sensitivity_No_ESCC  \
0          1.000000           0.02500  0.875399             1.000000   
1          0.788462           0.69863  0.788462             0.798077   
2          0.753247           0.81000  0.753247             0.740260   

   Specificity_No_ESCC  F1_No_ESCC  
0             0.000000    0.872611  
1  

Saved: RESULTS_FIGURES_177/Figure_3M_Temporal_Validation_bw.png

=== Temporal validation: 6M ===
Train n = 129
Test  n = 48
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 67, number of negative: 62
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 174
[LightGBM] [Info] Number of data points in the train set: 129, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

Saved: RESULTS_FIGURES_177/Figure_6M_Temporal_Validation_bw.png

=== Temporal validation: 12M ===
Train n = 129
Test  n = 48
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 50, number of negative: 79
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000060 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 174
[LightGBM] [Info] Number of data points in the train set: 129, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

Saved: RESULTS_FIGURES_177/Figure_12M_Temporal_Validation_bw.png

=== Temporal validation summary ===
  Horizon cutoff_date  n_train  n_test  test_events  AUC_temporal  \
0      3M  2018-12-31      129      48           40      0.642188   
1      6M  2018-12-31      129      48           37      0.769042   
2     12M  2018-12-31      129      48           27      0.826279   

   Brier_temporal  Calibration_slope  Calibration_intercept  
0        0.211648           0.051255               1.205206  
1        0.211924           0.086250               1.042350  
2        0.186634           0.119351               0.153950  


ストリーミング出力は最後の 5000 行に切り捨てられました。
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain